# CHB-MIT 单通道癫痫预测

`colab/` 文件夹放在 MyDrive 根目录，把这个笔记本上传到 Colab，挂载后按顺序跑。

- 代码：`/content/drive/MyDrive/colab`
- EDF：`/content/chbmit`（跑完一个患者删一个）
- 结果：`/content/drive/MyDrive/colab/results/{pid}/`

## 0. 挂载 Drive、导入

In [ ]:
!pip install -q mne


In [ ]:
import gc
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

CODE = Path("/content/drive/MyDrive/colab")
DATA = Path("/content/chbmit")
OUT = CODE / "results"

sys.path.insert(0, str(CODE))
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

import torch
import torch.nn as nn

import tools as t
import colab_tools as ct
from models import ConvBILSTMClassify

print("device =", t.DEVICE)

## 1. 超参

In [ ]:
ELECS = ["Fp1-F7", "Fp2-F8", "T7-P7", "T8-P8", "P7-O1", "P8-O2"]
PATIENTS = ["chb%02d" % i for i in range(1, 25)]
# PATIENTS = ["chb01"]                # 调试时只跑一个

SPH, SOP = 2 * 60, 30 * 60           # 发作前留白 2 min，发作前期 30 min
INTER_THR = 2 * 60 * 60              # 发作间期离发作至少 2 h
WIN_LEN, WIN_OL = 4, 2               # 4 s 窗口，50% 重叠
ENC_VAL, CLS_VAL = 0.1, 0.1
SEED = 0

CHAN_ATTEN, TEMP_ATTEN = False, True
HIDDEN, DROPOUT = 20, 0.7
LR, WD = 1e-4, 1e-3
BATCH, PATIENCE, EPOCHS = 32, 10, 30

CONFIG = dict(
    learning_rate=LR, weight_decay=WD, batch_size=BATCH, dropout=DROPOUT,
    patience=PATIENCE, epochs=EPOCHS, hidden_size=HIDDEN,
    channal_attention=CHAN_ATTEN, temporal_attention=TEMP_ATTEN,
)

## 2. 主流程：下载一个 → 训练 → 测试 → 存一个

跑一个患者：
1. `ct.download_patient` 下 EDF
2. `t.preprocess` 切窗打标签
3. `t.train_val_test` 按 LOOCV 切成张量
4. `t.train_elec_model` 六个电极各训一个 `ConvBILSTMClassify`
5. 写 manifest（权重路径 + 每折日志）
6. `t.train_elec_atten` 训练电极融合模型
7. `t.test_elec_model` 测试并把指标写回 manifest
8. 结果存 Drive，删本地 EDF

中途挂了就把 `PATIENTS` 改一下从断点继续，已下载的 EDF 会跳过。

In [ ]:
from concurrent.futures import ThreadPoolExecutor

for pid in PATIENTS:
    done = OUT / pid / ("%s.json" % pid)
    if done.exists():
        print("[跳过] 已完成", pid)
        continue
    print("=" * 15, pid, "=" * 15)
    out = OUT / pid
    weight_dir = out / "weight"
    weight_dir.mkdir(parents=True, exist_ok=True)

    # 1. 下载
    pdir = ct.download_patient(pid, DATA)

    # 2. 预处理 -> windows / meta
    windows, meta = t.preprocess(str(pdir), SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS)

    # 3. 划分（dataset 只建一份，六电极共用）
    dataset = t.train_val_test(ELECS, meta, windows, SEED, (ENC_VAL, CLS_VAL))
    del windows, meta
    gc.collect()

    # 4. 六个电极并行训练
    def train_one(i, elec):
        print("正在训练model：%s" % elec)
        model = ConvBILSTMClassify(channal_attention=CHAN_ATTEN, temporal_attention=TEMP_ATTEN, dropout=DROPOUT)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        loss_function = nn.BCEWithLogitsLoss()
        fold_paths, fold_logs = t.train_elec_model(model, optimizer, loss_function, dataset, i, ELECS,
                                                   weight_dir, patience=PATIENCE,
                                                   epochs=EPOCHS, batch_size=BATCH)
        return elec, fold_paths, fold_logs

    with ThreadPoolExecutor(max_workers=len(ELECS)) as ex:
        results = list(ex.map(lambda p: train_one(*p), enumerate(ELECS)))
    torch.cuda.empty_cache()
    gc.collect()

    # 5. 写 manifest
    manifest = {"config": CONFIG, "weights": [], "logs": {}}
    for elec, fold_paths, fold_logs in results:
        manifest["weights"].extend(fold_paths)
        manifest["logs"][elec] = fold_logs
    now = datetime.now().strftime("%y%m_%d_%H%M%S")
    path = weight_dir / ("%s_weights_manifest.json" % now)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=4, ensure_ascii=False, default=str)

    # 6. 电极融合
    fusion_models = t.train_elec_atten(dataset, path)

    # 7. 测试，指标写回 manifest
    t.test_elec_model(dataset, fusion_models, path=path)

    # 8. 存结果、删 EDF
    shutil.copy(path, out / ("%s.json" % pid))
    del dataset, fusion_models
    torch.cuda.empty_cache()
    gc.collect()
    ct.clear_patient(pid, DATA)
    print("[完成]", pid)


## 3. 汇总

In [ ]:
df = ct.summarize(OUT)
print("单电极 AUC: %.4f | 融合 AUC: %.4f" % (df.auc.mean(), df.auc_atten.mean()))
df